# 第14章 强化学习
## Reinforcement Learning — 试错中学习的艺术

**来源：李宏毅《深度学习教程》第14章 | 对应原书第236-255页**

---

## 一、知识地图：全章结构与脉络

```
第14章 强化学习
├── 14.1 强化学习应用
│   ├── 玩电子游戏（太空侵略者）
│   └── 下围棋（AlphaGo）
├── 14.2 强化学习框架（机器学习三步骤）
│   ├── 第1步：未知函数 = 策略网络
│   │   ├── 输入：观测（游戏画面）
│   │   ├── 输出：动作概率分布
│   │   └── 随机采样动作（非取最大）
│   ├── 第2步：定义损失
│   │   ├── 回合、轨迹、奖励、回报
│   │   └── 最大化总回报
│   └── 第3步：优化
│       ├── 轨迹定义
│       ├── 环境和奖励的随机性
│       └── 策略梯度
├── 14.3 评价动作的标准（5个版本进化）
│   ├── V0：即时奖励（短视）
│   ├── V1：累积奖励（不分远近）
│   ├── V2：折扣累积奖励（gamma，远小近大）
│   ├── V3：减基线（正负分明）
│   └── V3.5：Critic评价（知道期望值）
├── 14.3.5 Actor-Critic
│   ├── Actor（策略网络）：出动作
│   ├── Critic（价值网络）：评好坏
│   ├── MC vs TD训练方法
│   ├── 网络共享前几层
│   └── DQN（Rainbow）
├── 14.3.6 优势Actor-Critic
│   └── A_t = r_t + gamma * V(s_{t+1}) - V(s_t)
├── 同策略 vs 异策略
├── 探索 vs 利用
└── 强化学习的其他技巧
    ├── 稀疏奖励处理
    ├── 模仿学习
    └── 视觉强化学习 (VRL)
```

## 二、强化学习 vs 监督学习：根本区别

### 2.1 监督学习的局限

监督学习需要"正确答案"：给一张图，告诉机器"这是猫"。

但有些任务没有"正确答案"：
- 下围棋：给定一个盘势，最佳落子位置未知（人类的落子只是"好答案"，不一定是"最优"）
- 玩游戏：应该左移还是开火？没有标准答案

### 2.2 强化学习的核心思想

**Agent与环境互动，通过奖励信号来学习**：

```
环境 -> 观测s_t -> Agent -> 动作a_t -> 环境 -> 新观测s_{t+1} + 奖励r_t
```

| | 监督学习 | 强化学习 |
|------|---------|---------|
| 正确答案 | 已知 | **未知**（只有奖励信号） |
| 学习方式 | 模仿正确答案 | 尝试→得到奖励→调整策略 |
| 目标 | 最小化预测误差 | 最大化累积奖励 |

> **类比**：监督学习是"老师在旁告诉你每一步该怎么做"，强化学习是"把你扔到游戏里，赢了告诉你赢了，输了告诉你输了，你自己琢磨怎么赢"。

## 三、强化学习的三个步骤

### 第1步：定义未知函数 = 策略网络

策略网络 $\pi_\theta(a|s)$：给定状态 $s$，输出每个动作 $a$ 的概率。

**为什么随机采样而非取最大概率的动作？**

1. **探索需要**：如果总是出石头，在石头剪刀布中一定输
2. **随机性对很多游戏至关重要**：同样的画面，每次做不同的尝试才能发现好的策略

### 第2步：定义损失 = 最大化回报

**关键概念**：
- **回合（Episode）**：从游戏开始到结束
- **轨迹 $\tau = \{s_1, a_1, s_2, a_2, ..., s_T, a_T\}$**
- **奖励 $r_t$**：采取动作 $a_t$ 后立即得到的反馈
- **回报 $R(\tau) = \sum_{t=1}^T r_t$**：整个轨迹的总奖励

**目标**：最大化期望回报 $\mathbb{E}[R(\tau)]$

### 第3步：优化 = 策略梯度

强化学习不能用普通的梯度下降，因为：
1. 策略网络有随机性（输出是概率分布，动作为采样）
2. 环境和奖励是**黑盒子**（可能也有随机性）

**策略梯度公式**：

$$\nabla \bar{R}_\theta \approx \frac{1}{N}\sum_{n=1}^N \sum_{t=1}^{T_n} A_t \cdot \nabla \log \pi_\theta(a_t^n | s_t^n)$$

其中 $A_t$ 是动作 $a_t$ 的评价值（好坏分数）。

> **直觉**：$A_t > 0$（好动作）→ 增大执行该动作的概率；$A_t < 0$（坏动作）→ 减小执行该动作的概率。$|A_t|$ 越大，调整幅度越大。**这本质上是一个加权交叉熵**——权重就是评价值 $A_t$。

## 四、动作评价的五级进化

策略梯度的核心问题是：**如何定义 $A_t$（动作的好坏）？**

### V0：即时奖励 $A_t = r_t$（短视版）

$A_t$ 只看当前动作立即得到的奖励。

**问题**：太空侵略者中，左右移动奖励为0（只有开火命中才得分），用V0训练会导致智能体一直开火、从不移动。但实际上左右移动是瞄准的必要步骤！

### V1：累积奖励 $A_t = G_t = \sum_{i=t}^T r_i$

$A_t = $ 从$t$时刻开始到游戏结束的所有奖励之和。

**问题**：如果游戏很长，把最后一刻的奖励全部归功于最初的动作，不太合理。

### V2：折扣累积奖励 $A_t = G'_t = \sum_{i=t}^T \gamma^{i-t} r_i$

引入**折扣因子 $\gamma \in [0, 1]$**（通常0.9或0.99）。

$$G'_t = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \gamma^3 r_{t+3} + \cdots$$

越远的奖励权重越小（$\gamma^{距离}$迅速衰减）。

- $\gamma \to 1$：远视（看重长期）
- $\gamma \to 0$：近视（只看眼前）

### V3：减基线 $A_t = G'_t - b$

**问题**：如果游戏中所有动作的奖励都是正的（只是大小不同），V2会鼓励所有动作。

**解决**：减掉一个基线 $b$（如 $G'_t$ 的平均值），让 $A_t$ 有正有负。

### V3.5（Actor-Critic）：使用价值网络作为基线

让 $b = V^{\pi_\theta}(s_t)$ —— **Critic（价值网络）** 预测的"在状态 $s_t$ 下，接下来会得到的期望折扣累积奖励"。

$A_t = G'_t - V^{\pi_\theta}(s_t)$ > 0：这个动作比平均水平好！
$A_t = G'_t - V^{\pi_\theta}(s_t)$ < 0：这个动作比平均水平差！

> **直觉**：Critic告诉你"在这个状态下一般能得多少分"，实际得分减期望得分 = 该动作"超出预期"的程度。

In [ ]:
# ============================================
# PyTorch示例1：策略梯度计算（V2折扣奖励版）
# ============================================
import torch
import torch.nn as nn
import numpy as np

def compute_discounted_rewards(rewards, gamma=0.99):
    """
    计算折扣累积奖励 G'_t
    G'_t = r_t + gamma*r_{t+1} + gamma^2*r_{t+2} + ...
    """
    discounted = []
    running = 0
    for r in reversed(rewards):
        running = r + gamma * running
        discounted.insert(0, running)
    discounted = torch.tensor(discounted)
    # 标准化（减均值除标准差，稳定训练）
    discounted = (discounted - discounted.mean()) / (discounted.std() + 1e-8)
    return discounted

def policy_gradient_loss(log_probs, discounted_rewards):
    """
    策略梯度损失（V2版）
    L = -sum(A_t * log_prob(a_t|s_t))
    其中 A_t = 折扣累积奖励
    """
    return -(log_probs * discounted_rewards).sum()

print("策略梯度计算已实现（V2折扣奖励版）。")
print()
print("核心公式：G'_t = r_t + gamma*r_{t+1} + gamma^2*r_{t+2} + ...")
print("gamma=0.99时，gamma^100 ≈ 0.366，远距离奖励影响很小。")

## 五、Actor-Critic架构

### 5.1 两个网络，协同工作

| 网络 | 作用 | 输入 | 输出 |
|------|------|------|------|
| **Actor（策略网络）** | 决定做什么 | 状态 $s$ | 动作概率分布 |
| **Critic（价值网络）** | 评价状态好坏 | 状态 $s$ | 标量 $V(s)$ |

$V^{\pi_\theta}(s)$ 表示：在状态 $s$ 下，按照策略 $\pi_\theta$ 行动，接下来期望获得的折扣累积奖励。

### 5.2 训练Critic的两种方法

**蒙特卡洛 (MC)**：
1. 智能体玩完整场游戏，记录所有 $(s_t, G'_t)$
2. 训练Critic让 $V(s_t)$ 接近实际得到的 $G'_t$
3. 优点：无偏估计
4. 缺点：需要玩完整场游戏（有的游戏很长或永不结束），方差大

**时序差分 (TD)**：
1. 只需要一段数据 $(s_t, a_t, r_t, s_{t+1})$
2. 利用关系：$V(s_t) \approx r_t + \gamma V(s_{t+1})$
3. 训练Critic让 $V(s_t) - \gamma V(s_{t+1})$ 接近 $r_t$
4. 优点：可以在线学习，不需要等游戏结束，方差小
5. 缺点：有偏估计

**MC vs TD 的答案可能不同**（两者假设不同）：
- MC直接看观察到的数据 → "因为实际只得了0分，所以 $V(s_a)=0$"
- TD认为 $s_a$ 和 $s_b$ 无关 → "$s_b$ 的平均奖励是3/4，$s_a$ 的奖励也应该是3/4"

### 5.3 优势Actor-Critic (Advantage A2C) — V4

**核心改进**：用期望值替代采样值

$$A_t = r_t + \gamma V^{\pi_\theta}(s_{t+1}) - V^{\pi_\theta}(s_t)$$

- $r_t + \gamma V(s_{t+1})$：执行 $a_t$ 后的期望奖励
- $V(s_t)$：从 $s_t$ 出发的平均期望奖励
- 差值 > 0：这个动作比平均好
- 差值 < 0：这个动作比平均差

这比用采样值 $G'_t$ 更稳定（用期望代替单次采样的随机波动）。

In [ ]:
# ============================================
# PyTorch示例2：Actor-Critic架构
# ============================================
class ActorCritic(nn.Module):
    """
    Actor-Critic网络
    Actor和Critic共享前面的卷积/全连接层
    这利用了输入相同（都是状态s）的特点
    """
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        # 共享的特征提取层
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
        )
        
        # Actor头：输出动作分布
        self.actor_head = nn.Sequential(
            nn.Linear(hidden, action_dim),
            nn.Softmax(dim=-1)
        )
        
        # Critic头：输出状态价值
        self.critic_head = nn.Linear(hidden, 1)
    
    def forward(self, state):
        features = self.shared(state)
        action_probs = self.actor_head(features)  # [B, action_dim]
        state_value = self.critic_head(features)   # [B, 1]
        return action_probs, state_value
    
    def act(self, state):
        """给定状态，采样动作"""
        probs, value = self.forward(state.unsqueeze(0))
        action = torch.multinomial(probs, 1)
        return action.item(), probs[0, action].log(), value

print("Actor-Critic网络已定义。")
print()
print("Actor和Critic共享特征提取层，减少参数量。")
print("A2C优势函数：A_t = r_t + gamma*V(s_{t+1}) - V(s_t)")

## 六、强化学习的关键技巧

### 6.1 同策略 (On-Policy) vs 异策略 (Off-Policy)

| | 同策略 (On-Policy) | 异策略 (Off-Policy) |
|------|-------------------|---------------------|
| 交互智能体 | = 训练的智能体 | != 训练的智能体 |
| 数据采集 | 每次更新后重新采集 | 一次采集多次使用 |
| 效率 | 低（更新400次=采集400次） | 高 |
| 代表 | 策略梯度 | DQN, PPO |

> **类比（棋魂）**：弱小的进藤光（旧策略）觉得小马步飞是对的，强大的进藤光（新策略）应该下大马步飞。同一个动作对不同水平的智能体意义不同——所以旧策略的数据不能直接用来训练新策略。这是On-Policy每次都要重新采集数据的根本原因。

### 6.2 探索 vs 利用

**探索（Exploration）**：尝试新动作，发现可能更好的策略

**利用（Exploitation）**：执行已知最好的动作，最大化当前奖励

二者需要平衡：
- 如果从不探索 → 可能永远发现不了更好的动作
- 如果一直探索 → 无法累积奖励

**提高探索的方法**：
- 加大策略网络输出的熵（让分布更均匀）
- 在策略网络参数上加噪声
- $\epsilon$-贪心：以 $\epsilon$ 概率随机选动作

### 6.3 DQN与Rainbow

深度Q网络（DQN）：直接用Critic（价值网络）来选择动作（不用Actor）。

Rainbow：将DQN的七种改进方法组合在一起——因此叫"彩虹"。

### 6.4 稀疏奖励

围棋只有最后赢了才得1分，中间没有任何奖励——这叫稀疏奖励。

解决方法：设计中间奖励、好奇心驱动、层次化RL等。

### 6.5 视觉强化学习 (VRL)

直接从图像（像素）学习控制策略——不需要手工设计状态特征。这是当前RL的前沿方向。

## 七、强化学习与GAN的关系

强化学习和GAN有深刻的相似性：

| | RL | GAN |
|------|-----|------|
| 生成器 | 策略网络（Actor） | 生成器 G |
| 判别器/评价者 | 环境+奖励函数 | 判别器 D |
| 目标 | 最大化累积奖励 | 最大化 D 的评分 |
| 核心困难 | 环境和奖励不可微分 | 离散输出时梯度为零 |

> **共通点**：两者都需要最大化一个"黑盒子"的反馈。GAN用判别器，RL用环境——但它们都不是可微分的神经网络，所以都需要特殊的优化方法。

## 八、跨章节连接

| 章节 | 连接关系 |
|------|----------|
| Ch8 GAN | 训练GAN的生成器 = 训练RL的策略网络（都需要最大化来自外部的分数） |
| Ch11 自编码器 | 文字嵌入自编码器的解码器输出离散文字 → 需要RL方法训练 |
| Ch12 对抗攻击 | 攻击和防御可以建模为RL问题 |
| Ch13 迁移学习 | 从仿真环境迁移到真实世界（Sim-to-Real）是RL的重要挑战 |
| Ch15 元学习 | 元学习中的优化问题有时也用RL方法求解 |

## 九、核心要点总结

1. **RL的独特之处**：没有"正确答案"，只有"奖励信号"。Agent通过与环境的试错互动来学习。

2. **策略网络**：输入状态，输出动作概率分布。随机采样动作（而非取最大）对探索至关重要。

3. **策略梯度**：$\nabla \bar{R} \approx \sum A_t \cdot \nabla \log \pi(a_t|s_t)$——加权交叉熵，权重=动作评价值 $A_t$。

4. **动作评价五级进化**：即时奖励 → 累积 → 折扣累积($\gamma$) → 减基线 → Actor-Critic($V(s)$) → 优势A2C($r+\gamma V(s')-V(s)$)。

5. **Actor-Critic**：Actor出动作，Critic评好坏。两者可共享特征提取网络。

6. **MC vs TD**：MC需完整游戏无偏但方差大；TD可在线学习有偏但方差小。实践中TD更常用。

7. **同策略 vs 异策略**：同策略每次更新需重新采集数据（效率低）；异策略可复用人采集数据（效率高）。

8. **探索 vs 利用**：RL训练中的核心权衡。随机性是RL成功的关键——如果某些动作从未被尝试，它的好坏永远是未知的。

9. **延迟奖励**：很多任务中的关键动作（如瞄准）没有即时奖励，但却是获得最终奖励的必要条件。折扣累积奖励解决了这个问题。

10. **RL的困难**：训练极其耗时（每次更新参数后需重新采集数据），抽样质量对结果影响极大。RL是通向通用人工智能的可能途径之一。

## 十、练习与思考

1. 强化学习和监督学习的根本区别是什么？在什么情况下应该选择RL而非监督学习？

2. 为什么策略网络输出的是概率分布而非确定性动作？去掉随机性会怎样？

3. 推导策略梯度公式 $\nabla \bar{R}_\theta \approx \sum A_t \nabla \log \pi_\theta(a_t|s_t)$。为什么是 $\log \pi$ 而非 $\pi$？

4. 用"太空侵略者"的例子，分别说明V0（即时奖励）和V2（折扣累积）的优劣。为什么V0会让智能体一直开火？

5. 解释折扣因子 $\gamma$ 的作用。$\gamma = 0$ 和 $\gamma = 1$ 分别对应什么行为？

6. V3中为什么要减基线 $b$？如果所有奖励都是正的会有什么问题？

7. MC和TD方法在训练Critic时分别怎么计算？为什么它们可能给出不同的 $V(s)$？

8. 优势Actor-Critic中的 $A_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ 各项分别代表什么含义？

9. 解释同策略（On-Policy）和异策略（Off-Policy）的区别。为什么同策略每更新一次参数就要重新采集数据？

10. 探索不足会有什么后果？提高探索能力的常用方法有哪些？

11. RL和GAN在哪些方面相似？为什么两者都需要"特殊的优化方法"？